# 05 · SHAP — Galaxy Morphology Classification

**SHAP GradientExplainer** on the frozen AstroDINO + linear probe pipeline  
to identify which spatial regions of the galaxy image drive each morphology class.

| Step | Description |
|------|-------------|
| 1 | Load model, build `MorphDataset` (Spheroid / Disk / Bulge) |
| 2 | Extract embeddings, train linear probe |
| 3 | Wrap as `CombinedModel` (backbone → linear head) |
| 4 | `shap.GradientExplainer` → pixel attribution maps |
| 5 | Visualize: per-class mean |SHAP| + per-galaxy gallery |

In [ ]:
import os, sys, glob
import numpy as np
import h5py
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import transforms
from omegaconf import OmegaConf
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import shap
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

PROJECT_ROOT = '/home/yacheng/ssl_outthere'
BENCH_ROOT   = os.path.join(PROJECT_ROOT, 'encoder_image/astrodino/benchmark')
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, BENCH_ROOT)

from dinov2.eval.setup import build_model_for_eval
from preprocessing import get_torgb

DEG_TO_PIXEL = 3600 * 1000 / 30   # 30 mas/pixel
MORPH_NAMES  = {0: 'Spheroid', 1: 'Disk', 3: 'Bulge'}
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
MODEL_CONFIG  = f'{PROJECT_ROOT}/encoder_image/astrodino/model/astrodino_f150w_vitb_ps6_bs128/config.yaml'
MODEL_WEIGHTS = f'{PROJECT_ROOT}/encoder_image/astrodino/model/astrodino_f150w_vitb_ps6_bs128/eval/training_299999/teacher_checkpoint.pth'
DATA_ROOT     = f'{PROJECT_ROOT}/images/jwst/f150w'

REFF_MIN_PIX    = 2.3
REFF_MAX_PIX    = 72
N_PER_CLASS     = 2000   # samples per class for linear probe training
N_BACKGROUND    = 100    # SHAP background samples
N_EXPLAIN_CLASS = 10      # gallery examples per class
SEED            = 42
rng = np.random.default_rng(SEED)

## Model Loading

In [ ]:
cfg   = OmegaConf.load(MODEL_CONFIG)
model = build_model_for_eval(cfg, pretrained_weights=MODEL_WEIGHTS)
model = model.to(DEVICE).eval()
CROP_SIZE = cfg.crops.global_crops_size
TO_RGB, IN_CHANS = get_torgb(cfg)
print(f'Model loaded — crop={CROP_SIZE}  in_chans={IN_CHANS}  ToRGB={type(TO_RGB).__name__}')

## Dataset + Embeddings + Linear Probe

In [ ]:
class MorphDataset(Dataset):
    """Balanced Spheroid / Disk / Bulge dataset (Irregular excluded)."""
    def __init__(self, root, crop_size, reff_min, reff_max, n_per_class, seed=42):
        self.crop = transforms.CenterCrop(crop_size)
        self._rng = np.random.default_rng(seed)
        self._files = []
        for fp in sorted(glob.glob(os.path.join(root, '*.h5'))):
            f = h5py.File(fp, 'r')
            if 'morph_flag_f150w' in f: self._files.append(f)
            else: f.close()

        self._idx = []
        for fi, f in enumerate(self._files):
            morph = f['morph_flag_f150w'][:]
            re    = f['radius_sersic'][:] * DEG_TO_PIXEL if 'radius_sersic' in f else None
            vmask = np.isfinite(morph) & (morph != 2)
            if re is not None:
                vmask &= np.isfinite(re) & (re >= reff_min) & (re <= reff_max)
            for li in np.where(vmask)[0]:
                self._idx.append((fi, li, int(morph[li])))

        by_cls = {}
        for i, (_, _, lbl) in enumerate(self._idx):
            by_cls.setdefault(lbl, []).append(i)
        n = min(n_per_class, min(len(v) for v in by_cls.values()))
        kept = []
        for lbl, idxs in sorted(by_cls.items()):
            chosen = self._rng.choice(idxs, n, replace=False)
            kept.extend(chosen.tolist())
        self._rng.shuffle(kept)
        self._idx = [self._idx[i] for i in kept]
        print(f'Dataset: {len(self._idx)} samples ({n}/class) — {list(by_cls.keys())}')

    def __len__(self): return len(self._idx)

    def __getitem__(self, i):
        fi, li, lbl = self._idx[i]
        img = self._files[fi]['image'][li].astype('float32')
        img = np.repeat(img[np.newaxis], IN_CHANS if IN_CHANS > 1 else 1, axis=0)
        t   = self.crop(torch.from_numpy(img))
        t   = torch.from_numpy(TO_RGB(t.numpy()))
        return t, lbl

    def close(self):
        for f in self._files:
            try: f.close()
            except: pass

ds = MorphDataset(DATA_ROOT, CROP_SIZE, REFF_MIN_PIX, REFF_MAX_PIX, N_PER_CLASS, SEED)

In [ ]:
# ── Extract embeddings and keep raw image tensors for SHAP ────────────────────
loader = DataLoader(ds, batch_size=256, shuffle=False, num_workers=4, pin_memory=True)
emb_list, lbl_list, img_list = [], [], []
with torch.no_grad():
    for imgs, lbls in tqdm(loader, desc='Embedding'):
        e = model(imgs.to(DEVICE))
        if isinstance(e, tuple): e = e[0]
        if e.dim() > 2: e = e.view(e.size(0), -1)
        emb_list.append(e.cpu().numpy())
        lbl_list.append(lbls.numpy() if isinstance(lbls, torch.Tensor) else np.array(lbls))
        img_list.append(imgs.cpu())
ds.close()

emb_all = np.concatenate(emb_list)
lbl_all = np.concatenate(lbl_list)
img_all = torch.cat(img_list)    # (N, C, H, W) preprocessed float32
print(f'Embeddings: {emb_all.shape}  Images: {tuple(img_all.shape)}')

# ── Train / test split ────────────────────────────────────────────────────────
all_idx = np.arange(len(emb_all))
tr_idx, te_idx = train_test_split(all_idx, test_size=0.2, stratify=lbl_all, random_state=SEED)
X_tr, X_te = emb_all[tr_idx], emb_all[te_idx]
y_tr, y_te = lbl_all[tr_idx], lbl_all[te_idx]
img_tr, img_te = img_all[tr_idx], img_all[te_idx]

# ── Linear probe ──────────────────────────────────────────────────────────────
classes    = sorted(np.unique(y_tr).tolist())   # [0, 1, 3]
cls_to_id  = {c: i for i, c in enumerate(classes)}

head = nn.Linear(X_tr.shape[1], len(classes)).to(DEVICE)
opt  = torch.optim.Adam(head.parameters(), lr=5e-4, weight_decay=1e-4)
crit = nn.CrossEntropyLoss()

y_tr_t = torch.tensor([cls_to_id[c] for c in y_tr], dtype=torch.long)
tr_loader = DataLoader(TensorDataset(torch.tensor(X_tr, dtype=torch.float32), y_tr_t),
                       batch_size=128, shuffle=True)
for _ in tqdm(range(50), desc='Linear probe'):
    head.train()
    for xb, yb in tr_loader:
        opt.zero_grad()
        crit(head(xb.to(DEVICE)), yb.to(DEVICE)).backward()
        opt.step()

head.eval()
with torch.no_grad():
    logits = head(torch.tensor(X_te, dtype=torch.float32).to(DEVICE)).cpu().numpy()
y_pred = np.array([classes[i] for i in logits.argmax(axis=1)])

acc = accuracy_score(y_te, y_pred)
f1  = f1_score(y_te, y_pred, average='weighted')
print(f'\nLinear probe: acc={acc*100:.2f}%  F1={f1:.4f}')

## SHAP GradientExplainer

`CombinedModel` wraps backbone + linear head so SHAP can attribute pixel contributions  
all the way from raw (preprocessed) image to class logit.

In [ ]:
class CombinedModel(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head

    def forward(self, x):
        e = self.backbone(x)
        if isinstance(e, tuple): e = e[0]
        if e.dim() > 2: e = e.view(e.size(0), -1)
        return self.head(e)

combined = CombinedModel(model, head).to(DEVICE)
combined.eval()
print(f'CombinedModel: ({IN_CHANS},{CROP_SIZE},{CROP_SIZE}) → {len(classes)} classes {[MORPH_NAMES[c] for c in classes]}')

In [ ]:
# Background: random sample from training images
bg_idx     = rng.choice(len(img_tr), size=N_BACKGROUND, replace=False)
bg_tensors = img_tr[bg_idx].to(DEVICE).float()

# Explain: correctly-classified examples per class
explain_imgs, explain_lbls = [], []
for cls in classes:
    correct = np.where((y_te == cls) & (y_pred == cls))[0][:N_EXPLAIN_CLASS]
    explain_imgs.append(img_te[correct])
    explain_lbls.extend([cls] * len(correct))
    print(f'  {MORPH_NAMES[cls]}: {len(correct)} examples')

explain_tensors = torch.cat(explain_imgs, dim=0).to(DEVICE).float()
explain_lbls    = np.array(explain_lbls)
print(f'\nBackground: {bg_tensors.shape}  Explain: {explain_tensors.shape}')

In [ ]:
# GradientExplainer uses Expected Gradients — works with any differentiable model incl. ViT
# nsamples controls gradient estimation quality vs speed; 50 is fast (~1 min on GPU)
explainer = shap.GradientExplainer(combined, bg_tensors)
shap_vals = explainer.shap_values(explain_tensors, nsamples=10)

# Normalise to (N, n_cls, H, W) regardless of SHAP version:
#   newer SHAP returns (N, C, H, W, n_cls)  — class axis last
#   older SHAP returns list of n_cls arrays each (N, C, H, W)
raw = np.array(shap_vals) if not isinstance(shap_vals, np.ndarray) else shap_vals
print(f'Raw shap shape: {raw.shape}')

if raw.shape[-1] == len(classes):           # (N, C, H, W, n_cls)
    shap_np = raw.squeeze(1).transpose(0, 3, 1, 2)   # → (N, n_cls, H, W)
elif raw.shape[1] == len(classes):          # (N, n_cls, C, H, W) or (N, n_cls, H, W)
    shap_np = raw.squeeze(2) if raw.ndim == 5 else raw
elif raw.shape[0] == len(classes):          # (n_cls, N, C, H, W)
    shap_np = raw.squeeze(2).transpose(1, 0, 2, 3)
else:
    raise ValueError(f'Unexpected shap shape {raw.shape}')

explain_np = explain_tensors.cpu().numpy()[:, 0]     # (N, H, W)
print(f'shap_np: {shap_np.shape}  explain_np: {explain_np.shape}')
# Expected: (12, 3, 72, 72) and (12, 72, 72)

## Visualisation

In [ ]:
# Mean |SHAP| per class — averaged over correctly-classified examples of that class
fig, axes = plt.subplots(1, len(classes), figsize=(4.5 * len(classes), 4.2))

for ax, cls in zip(axes, classes):
    ci        = cls_to_id[cls]
    cls_mask  = (explain_lbls == cls)
    mean_shap = np.abs(shap_np[cls_mask, ci]).mean(axis=0)   # (H, W)
    mean_img  = explain_np[cls_mask].mean(axis=0)             # (H, W)

    ax.imshow(mean_img, cmap='gray', origin='lower')
    im = ax.imshow(mean_shap, cmap='inferno', alpha=0.65, origin='lower')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(MORPH_NAMES[cls], fontsize=14)
    ax.axis('off')

fig.suptitle('Mean |SHAP| per morphology class  (averaged over correctly-classified examples)',
             fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Gallery: per class — pairs of (image, SHAP overlay for that class)
n_row = len(classes)
n_col = N_EXPLAIN_CLASS
fig, axes = plt.subplots(n_row, n_col * 2, figsize=(n_col * 3.2, n_row * 2.8))

for row, cls in enumerate(classes):
    ci       = cls_to_id[cls]
    cls_idxs = np.where(explain_lbls == cls)[0]

    for col, idx in enumerate(cls_idxs[:n_col]):
        img  = explain_np[idx]          # (H, W)
        sv   = shap_np[idx, ci]         # SHAP toward true/predicted class (H, W)
        vmax = np.percentile(np.abs(sv), 97)

        ax_i = axes[row, col * 2]
        ax_i.imshow(img, cmap='gray', origin='lower')
        ax_i.axis('off')
        if col == 0:
            ax_i.set_ylabel(MORPH_NAMES[cls], fontsize=12)

        ax_s = axes[row, col * 2 + 1]
        ax_s.imshow(img, cmap='gray', origin='lower')
        ax_s.imshow(sv, cmap='RdBu_r', alpha=0.65,
                    vmin=-vmax, vmax=vmax, origin='lower')
        ax_s.axis('off')

# Column headers on top row
for col in range(n_col):
    axes[0, col * 2    ].set_title('Image', fontsize=9)
    axes[0, col * 2 + 1].set_title('SHAP',  fontsize=9)

fig.suptitle('Galaxy morphology SHAP attribution  (red = pushes toward class, blue = away)',
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Radial SHAP profile — how attribution strength varies with distance from galaxy centre
H, W = shap_np.shape[-2], shap_np.shape[-1]
cy, cx = H // 2, W // 2
yy, xx = np.ogrid[:H, :W]
r_map   = np.sqrt((yy - cy)**2 + (xx - cx)**2).ravel()   # (H*W,)
r_bins  = np.arange(0, r_map.max() + 1)

fig, ax = plt.subplots(figsize=(7, 4))
for cls in classes:
    ci       = cls_to_id[cls]
    cls_mask = (explain_lbls == cls)
    abs_shap = np.abs(shap_np[cls_mask, ci]).mean(axis=0).ravel()   # (H*W,)
    profile  = [abs_shap[r_map.astype(int) == r].mean() if (r_map.astype(int) == r).any() else 0
                for r in r_bins]
    ax.plot(r_bins, profile, label=MORPH_NAMES[cls], lw=2)
ax.set_xlim(-1,20)
ax.set_xlabel('Radial distance from centre [pixels]')
ax.set_ylabel('Mean |SHAP|')
ax.set_title('Radial SHAP profile per morphology class')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Patch-Level SHAP

Replace the 72×72=5184-pixel input with **144 patch features** (one 6×6 block each).
Masked patches are filled with the training-set mean image.
`shap.KernelExplainer` computes Shapley values for each patch without gradient or ViT-specific assumptions.

| | Pixel SHAP | Patch SHAP |
|-|-----------|------------|
| Input dim | 5 184 | 144 |
| Method | GradientExplainer | KernelExplainer |
| Resolution | 72×72 (blocky) | 12×12 (native) |

In [ ]:
# ── Patch-level SHAP setup ────────────────────────────────────────────────────
PATCH_SIZE_P = 6
N_PATCHES_P  = CROP_SIZE // PATCH_SIZE_P   # 12

combined.eval()

# Background fill: mean of training images
bg_img = img_tr.float().mean(dim=0, keepdim=True).to(DEVICE)   # (1, C, H, W)

def patch_predict(masks):
    """
    masks  : (N, N_patches^2) array — 1 = keep original patch, 0 = fill with bg
    Returns: (N, n_classes) logits
    Uses '_current_img' (1, C, H, W) from enclosing scope.
    """
    masks_np = np.array(masks, dtype=np.float32)
    results  = []
    for i in range(0, len(masks_np), 128):
        batch    = masks_np[i:i+128]
        masks_up = np.stack([
            np.kron(m.reshape(N_PATCHES_P, N_PATCHES_P),
                    np.ones((PATCH_SIZE_P, PATCH_SIZE_P), dtype=np.float32))
            for m in batch
        ])                                                    # (bs, 72, 72)
        m_t    = torch.tensor(masks_up[:, None]).float().to(DEVICE)
        img_   = _current_img.expand(len(batch), -1, -1, -1)
        bg_    = bg_img.expand(len(batch), -1, -1, -1)
        masked = img_ * m_t + bg_ * (1 - m_t)
        with torch.no_grad():
            results.append(combined(masked).cpu().numpy())
    return np.concatenate(results, axis=0)

# KernelExplainer baseline: full unmasked image (all-ones mask)
bg_full = np.ones((1, N_PATCHES_P ** 2))

patch_shap_raw = []
for i in tqdm(range(len(explain_tensors)), desc='Patch SHAP'):
    _current_img = explain_tensors[i:i+1].float().to(DEVICE)
    explainer    = shap.KernelExplainer(patch_predict, bg_full)
    sv = explainer.shap_values(np.ones((1, N_PATCHES_P**2)), nsamples=500, silent=True)
    # This SHAP version returns (n_samples, n_features, n_classes) = (1, 144, 3)
    # Normalise to (n_cls, n_features) = (3, 144)
    if isinstance(sv, list):
        arr = np.stack([np.asarray(s).reshape(-1) for s in sv])  # (n_cls, 144)
    else:
        sv_np = np.asarray(sv).squeeze(0)   # remove sample dim → (144, n_cls)
        if sv_np.ndim == 2 and sv_np.shape[-1] == len(classes):
            arr = sv_np.T                   # (n_cls, 144)
        elif sv_np.ndim == 2 and sv_np.shape[0] == len(classes):
            arr = sv_np                     # already (n_cls, 144)
        else:
            arr = sv_np.reshape(len(classes), -1)
    patch_shap_raw.append(arr)

patch_shap_np   = np.array(patch_shap_raw)                   # (N, n_cls, 144)
patch_shap_maps = patch_shap_np.reshape(-1, len(classes),
                                         N_PATCHES_P, N_PATCHES_P)
print(f'Patch SHAP maps: {patch_shap_maps.shape}  (N, n_cls, 12, 12)')


In [ ]:
# ── Gallery: patch SHAP overlay ───────────────────────────────────────────────
fig, axes = plt.subplots(len(classes), N_EXPLAIN_CLASS * 2,
                         figsize=(N_EXPLAIN_CLASS * 3.2, len(classes) * 2.8))

for row, cls in enumerate(classes):
    ci       = cls_to_id[cls]
    cls_idxs = np.where(explain_lbls == cls)[0]
    for col, idx in enumerate(cls_idxs[:N_EXPLAIN_CLASS]):
        img   = explain_np[idx]
        sv    = patch_shap_maps[idx, ci]                     # (12, 12)
        sv_up = np.kron(sv, np.ones((PATCH_SIZE_P, PATCH_SIZE_P)))  # (72, 72)
        vmax  = np.percentile(np.abs(sv_up), 97)

        ax_i = axes[row, col * 2]
        ax_i.imshow(img, cmap='gray', origin='lower')
        ax_i.axis('off')
        if col == 0:
            ax_i.set_ylabel(MORPH_NAMES[cls], fontsize=12)

        ax_s = axes[row, col * 2 + 1]
        ax_s.imshow(img, cmap='gray', origin='lower')
        ax_s.imshow(sv_up, cmap='RdBu_r', alpha=0.65,
                    vmin=-vmax, vmax=vmax, origin='lower')
        ax_s.axis('off')

for col in range(N_EXPLAIN_CLASS):
    axes[0, col * 2    ].set_title('Image',      fontsize=9)
    axes[0, col * 2 + 1].set_title('Patch SHAP', fontsize=9)

fig.suptitle(
    f'Patch-level SHAP  ({PATCH_SIZE_P}x{PATCH_SIZE_P} px / feature, {N_PATCHES_P}x{N_PATCHES_P} grid)'
    '  red = supports class   blue = opposes class',
    fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Mean |patch SHAP| per class ───────────────────────────────────────────────
fig, axes = plt.subplots(1, len(classes), figsize=(4.5 * len(classes), 4.2))

for ax, cls in zip(axes, classes):
    ci       = cls_to_id[cls]
    cls_mask = (explain_lbls == cls)
    mean_sv  = np.abs(patch_shap_maps[cls_mask, ci]).mean(axis=0)   # (12, 12)
    mean_up  = np.kron(mean_sv, np.ones((PATCH_SIZE_P, PATCH_SIZE_P)))
    mean_img = explain_np[cls_mask].mean(axis=0)

    ax.imshow(mean_img, cmap='gray', origin='lower')
    im = ax.imshow(mean_up, cmap='inferno', alpha=0.65, origin='lower')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(MORPH_NAMES[cls], fontsize=14)
    ax.axis('off')

fig.suptitle('Mean |patch SHAP| per morphology class', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Pixel SHAP vs Patch SHAP comparison ───────────────────────────────────────
fig, axes = plt.subplots(len(classes), 3, figsize=(10, 3.0 * len(classes)))

for row, cls in enumerate(classes):
    ci  = cls_to_id[cls]
    idx = np.where(explain_lbls == cls)[0][0]
    img = explain_np[idx]

    px_sv  = shap_np[idx, ci]                                         # (72, 72)
    pt_sv  = np.kron(patch_shap_maps[idx, ci],
                     np.ones((PATCH_SIZE_P, PATCH_SIZE_P)))           # (72, 72)

    for col, (sv, title) in enumerate([
        (None,  'Image'),
        (px_sv, 'Pixel SHAP (GradientExplainer)'),
        (pt_sv, 'Patch SHAP (KernelExplainer)'),
    ]):
        ax = axes[row, col]
        ax.imshow(img, cmap='gray', origin='lower')
        if sv is not None:
            vmax = np.percentile(np.abs(sv), 97)
            ax.imshow(sv, cmap='RdBu_r', alpha=0.65,
                      vmin=-vmax, vmax=vmax, origin='lower')
        ax.axis('off')
        if row == 0:
            ax.set_title(title, fontsize=10)
        if col == 0:
            ax.set_ylabel(MORPH_NAMES[cls], fontsize=11)

fig.suptitle('Pixel SHAP vs Patch SHAP — one example per class', fontsize=12)
plt.tight_layout()
plt.show()